In [15]:
import gymnasium as gym
import stable_baselines3 as sb3
from gymnasium.utils.env_checker import check_env

In [16]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

class CustomEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.size = 10.0
        # Osservazioni come Dict
        self.observation_space = spaces.Dict({
            'height': spaces.Box(0, self.size, (1,), np.float32),
            'width':  spaces.Box(0, self.size, (1,), np.float32),
            'posx':   spaces.Box(-self.size, self.size, (1,), np.float32),
            'posy':   spaces.Box(-self.size, self.size, (1,), np.float32),
            'material': spaces.Discrete(3),
            'precision': spaces.Box(-1000, 1000, (1,), np.float32)
        })
        # Azioni come Dict (quelle che PPO non digerisce)
        self.action_space = spaces.Dict({
            'height': spaces.Box(0, self.size, (1,), np.float32),
            'width':  spaces.Box(0, self.size, (1,), np.float32),
            'posx':   spaces.Box(-self.size, self.size, (1,), np.float32),
            'posy':   spaces.Box(-self.size, self.size, (1,), np.float32),
            'material': spaces.Discrete(3)
        })

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs = {k: v.sample() if isinstance(v, spaces.Box) else 0 for k, v in self.observation_space.items()}
        return obs, {}

    def step(self, action):
        # Qui action arriva come DIZIONARIO grazie al wrapper sotto
        h, w = action['height'][0], action['width'][0]
        m = int(action['material'])
        
        reward = -np.abs(h - 5.0) + (10 if m == 2 else 0)
        terminated = bool(np.abs(h - 5.0) < 0.5 and m == 2)
        
        obs, _ = self.reset() # Semplificato per l'esempio
        return obs, float(reward), terminated, False, {}

# 2. IL WRAPPER "TRADUTTORE"
class ActionWrapper(gym.ActionWrapper):
    def __init__(self, env):
        super().__init__(env)
        # Sovrascriviamo lo spazio delle azioni dell'ambiente con un unico Box per PPO
        # [height, width, posx, posy, material]
        self.action_space = spaces.Box(
            low=np.array([0, 0, -10, -10, 0], dtype=np.float32),
            high=np.array([10, 10, 10, 10, 2.99], dtype=np.float32),
            dtype=np.float32
        )

    def action(self, action):
        # Converte il vettore di PPO nel dizionario per l'ambiente
        return {
            'height': np.array([action[0]], dtype=np.float32),
            'width':  np.array([action[1]], dtype=np.float32),
            'posx':   np.array([action[2]], dtype=np.float32),
            'posy':   np.array([action[3]], dtype=np.float32),
            'material': int(np.floor(action[4]))
        }

In [17]:

# 3. TRAINING
env = CustomEnv()
env = ActionWrapper(env) # Applichiamo la magia qui

model = PPO("MultiInputPolicy", env, verbose=1)
model.learn(total_timesteps=50000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-----------------------------
| time/              |      |
|    fps             | 1526 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn